# NBA Best Ball V2.9 — Original-Source Custody (R9-S19A)

Research-only. Upload and run the **unchanged original S19A** collector from the portable bundle. Run cells manually; **no downloads happen when this notebook opens**. Original responses are not independently timestamp witnessed and Google Drive is not WORM storage.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # Explicit authorization required for persistent source storage.


In [ ]:
from google.colab import files
import pathlib, zipfile, shutil
uploaded = files.upload()  # Upload NBA_V29_S19A_ALT_RUNNER_GITHUB_COLAB_20261014.zip
assert len(uploaded) == 1, 'Upload exactly one runner bundle ZIP'
name, data = next(iter(uploaded.items()))
assert name.endswith('.zip')
workspace=pathlib.Path('/content/nba_v29_runner')
workspace.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(name) as z:
    bad=[p for p in z.namelist() if pathlib.PurePosixPath(p).is_absolute() or '..' in pathlib.PurePosixPath(p).parts]
    assert not bad, f'Unsafe archive members: {bad}'
    z.extractall(workspace)
print('Runner extracted to', workspace)


In [ ]:
import subprocess, sys
res=subprocess.run([sys.executable,'-m','unittest','-q','test_source_gates','test_batch_capture'],cwd=str(workspace),capture_output=True,text=True)
print(res.stdout, res.stderr)
assert res.returncode == 0, 'Validation failed; do not run capture'


## Capture location
This writes the **exact original response bytes and local SHA-256 receipts** to your private Google Drive (subject to Drive mount limitations). Run the three-request smoke test first.


In [ ]:
import pathlib
capture_root=pathlib.Path('/content/drive/MyDrive/NBA Best Ball/V2.9 Source Capture R9-S19A')
capture_root.mkdir(parents=True, exist_ok=True)
print('Capture root:',capture_root)


In [ ]:
import subprocess, sys
smoke=subprocess.run([sys.executable,'batch_capture.py','--mode','smoke','--out',str(capture_root),'--delay','1.0'],cwd=str(workspace),text=True,capture_output=True)
print(smoke.stdout)
print(smoke.stderr)
if smoke.returncode:
    print('SMOKE FAILED — inspect FAILURES.ndjson; do not proceed with batch acquisition.')


## Optional batch acquisition — only after checking smoke results
Edit `start_index` and `count` for bounded source acquisition. Every call is user-invoked. Avoid exceeding publisher rate limits.


In [ ]:
# Explicitly set the flag to True before running a batch. Default is off.
APPROVE_BATCH = False
start_index, count = 0, 25
assert APPROVE_BATCH, 'Review smoke receipts first, then opt in manually'
assert 0 <= start_index <= 2461 and 1 <= count <= 100
batch=subprocess.run([sys.executable,'batch_capture.py','--mode','batch','--start-index',str(start_index),'--count',str(count),'--out',str(capture_root),'--delay','1.0'],cwd=str(workspace),text=True,capture_output=True)
print(batch.stdout)
print(batch.stderr)
